# 1 · SAT probability model

I'm starting with the uncertainty in the answers themselves: how many questions might be correct, given different confidence levels and guessing patterns? This notebook contains the model, the reasoning behind it, and a first experiment with converting counts into SAT points.

The examples use synthetic inputs. The answer-count calculation is exact under the assigned probabilities and independence assumptions. The SAT-point conversion is still experimental and hasn't been checked against real scores.

The sensitivity analysis below compares different assumed easier-route caps. Next, I plan to compare confidence probabilities, as described in [my progress notes](../PROGRESS.md).

I use the **SAT Lab (Python)** kernel and run the cells in order.

References: [College Board scoring](https://satsuite.collegeboard.org/scores/what-scores-mean/how-scores-calculated) and the [technical manual](https://research.collegeboard.org/media/pdf/Digital%20SAT%20Suite%20of%20Assessments%20Technical%20Manual-FINAL.pdf).

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the project folder or in notebooks/.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "satlab").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dataclasses import replace
from satlab.model import (Counts, Settings, poisson_binomial, summarize,
                          predict_test, predict_section)
from satlab.examples import example_modules


## 1. Modeling one answer

I define $X_i=1$ when question $i$ is correct and $0$ otherwise, with $P(X_i=1)=p_i$.
For a uniform guess between $k$ remaining choices, I assign $p_i=1/k$, conditional on the correct answer still being among those choices. Incorrect elimination would make that assignment overconfident.

The expected value follows directly:

$$E[X_i]=0(1-p_i)+1p_i=p_i,\qquad E[S]=\sum_i p_i.$$

Linearity of expectation does not require independence. For ten two-choice guesses, the expected count is 5. Under independence, the variance is $10(0.5)(0.5)=2.5$. The following calculation checks both values.

In [ ]:
probabilities = [0.5] * 10
distribution = poisson_binomial(probabilities)
stats = summarize(distribution)
display(pd.Series(stats, name="10 two-choice guesses"))
assert abs(stats["mean"] - 5) < 1e-10
assert abs(stats["variance"] - 2.5) < 1e-10


## 2. Calculating the number of correct answers

Questions can have different probabilities of being correct. Under independence, their sum $S=\sum_i X_i$ follows a **Poisson-binomial distribution**.
I write $f_i(s)=P(X_1+\cdots+X_i=s)$, starting from $f_0(0)=1$ and zero elsewhere.

$$f_i(s)=(1-p_i)f_{i-1}(s)+p_i f_{i-1}(s-1).$$

There are two disjoint ways to reach $s$ correct answers after question $i$: previously have $s$ and miss the next answer, or previously have $s-1$ and answer correctly. Independence justifies multiplying each previous probability by the corresponding answer probability; total probability justifies adding the two cases.

I implement this recurrence in `satlab/model.py` using dynamic programming. It calculates the finite distribution directly, without simulation.
An equivalent formulation takes the coefficient of $z^s$ in $\prod_i[(1-p_i)+p_i z]$.

Under independence, $\operatorname{Var}(S)=\sum_i p_i(1-p_i)$. Shared misconceptions or correlated errors can violate this assumption and change the actual uncertainty.

In [ ]:
mixed = [0.99] * 10 + [0.5] * 4 + [1/3] * 2 + [0.25]
pmf = poisson_binomial(mixed)
fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(pmf.keys(), pmf.values(), color="#26715a")
ax.set(xlabel="Correct answers", ylabel="Probability", title="Different guesses → a distribution, not one count")
plt.show()
assert abs(sum(pmf.values()) - 1) < 1e-10


## 3. Module inputs

I organize inputs by section and module: 27 questions per Reading & Writing module and 22 per Math module. Every question belongs to exactly one category.

- `confident`: believed correct, using a shared confidence probability.
- `guess2`, `guess3`, `guess4`: the number of choices remaining after elimination.
- `fill_right`, `fill_wrong`: Math responses verified against an answer key.
- `fill_uncertain`: Math responses assigned an explicit probability.
- `wrong`: other known wrong answers or unanswered questions.

I use 99% as the default probability for a confident answer. I haven't measured how well reported confidence matches correctness yet, so this is an assumption. A value of 1 would assume certainty.

The example below uses synthetic counts. I don't need to know which guesses were correct in order to calculate the possible outcomes.

In [ ]:
# Synthetic example inputs; each module accounts for its full question count.
modules = {
    "rw": [
        Counts(confident=19, guess2=4, guess3=2, guess4=1, wrong=1),
        Counts(confident=16, guess2=5, guess3=3, guess4=2, wrong=1),
    ],
    "math": [
        Counts(confident=14, guess2=2, guess3=1, guess4=1,
               fill_right=2, fill_wrong=1, fill_uncertain=1),
        Counts(confident=12, guess2=3, guess3=1, guess4=1,
               fill_right=2, fill_wrong=1, fill_uncertain=1, wrong=1),
    ],
}
settings = Settings(confidence=0.99, fill_probability=0.25, coverage=0.90)
result = predict_test(**modules, settings=settings)
display(pd.DataFrame({name: data["raw_summary"] for name, data in result["sections"].items()}).T)
print("SAT score conversion:", result["score_status"])


## 4. Interpreting low, median, and high

I define the quantile $q_\alpha=\min\{x:P(S\leq x)\geq\alpha\}$.
For a central 90% interval, I report $q_{.05}$, $q_{.50}$, and $q_{.95}$ as the likely low, median, and likely high outcomes.

These are quantiles rather than absolute minimum and maximum values. Because the distribution is discrete, the interval can contain more than 90% probability. It is a predictive interval conditional on the model, rather than a confidence interval for an unknown parameter.

Increasing the selected coverage cannot narrow this central interval, although the endpoints may stay fixed because answer counts are integers.

## 5. Pretest questions and Module 2 routing

Each module contains two unscored pretest questions. I approximate their unknown positions as a uniformly selected pair.
If $S=s$ of $n$ displayed questions are correct, and $U$ counts the correct answers in the pretest pair, then

$$P(U=u\mid S=s)=\frac{\binom{s}{u}\binom{n-s}{2-u}}{\binom{n}{2}}.$$

The numerator counts pairs containing $u$ correct and $2-u$ incorrect positions. The denominator counts all pairs. The resulting scored count is $s-u$.
Real pretest placement has restrictions, so uniform pairing is a simplifying assumption.

For an unknown Module 2 route, I use the illustrative function

$$P(H\mid s_1)=\operatorname{sigmoid}((s_1/n-t)/w).$$

Here $H$ denotes the harder route, $t$ is an assumed midpoint, and $w$ controls the transition width. This links routing to Module 1 performance. It is not College Board's routing algorithm or an established correct-answer threshold.

I sum over possible counts and routes using the law of total probability. An independently known route can instead be supplied as `hard` or `easy`.

## 6. Experimental SAT score conversion

The official SAT uses calibrated item characteristics. Counts alone do not recover that scoring process, so this conversion remains an **uncalibrated scenario model**.

I use the scored fraction $f=(s_1+s_2)/(2n)$ and the linear mapping

$$\text{base}=\text{floor}+(\text{cap}-\text{floor})f.$$

The easier route uses floor 200 and an assumed cap; the harder route uses an assumed floor and cap 800.
I add symmetric triangular noise in 10-point steps, round the result, and clip it to the route's limits. The default easier-route cap of 600 and the noise radius are illustrative parameters, not established SAT scoring facts.

I obtain the total distribution by convolving the two section distributions under assumed section independence, then calculate its quantiles. Adding section endpoint quantiles would not generally produce the same total interval.

The code requires `include_illustrative_scores=True` to make this conversion explicit.

In [ ]:
# Explicit opt-in to the uncalibrated SAT-point scenario.
routes = {"rw": "unknown", "math": "unknown"}  # or "hard" / "easy"
illustration = predict_test(**modules, routes=routes, settings=settings,
                            include_illustrative_scores=True)
rows = {name: data["score_summary"] for name, data in illustration["sections"].items()}
rows["total"] = illustration["total_score_summary"]
display(pd.DataFrame(rows).T[["low", "median", "high", "interval_mass"]])
print("Status:", illustration["score_status"], "— not a calibrated SAT forecast")

pmf = illustration["total_score_pmf"]
s = illustration["total_score_summary"]
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(pmf.keys(), pmf.values(), width=8, color="#26715a")
ax.axvspan(s["low"], s["high"], color="#a8be65", alpha=.25, label="Selected model interval")
ax.axvline(s["median"], color="#163d33", linestyle="--", label="Median")
ax.set(xlabel="Illustrative total SAT points", ylabel="Probability", title="Uncalibrated scenario — assumptions matter")
ax.legend()
plt.show()


## 7. Sensitivity to the assumed score cap

I compare easier-route caps of 550, 600, and 650 while holding the other settings fixed. This shows how an unverified conversion parameter affects the resulting interval.

This helps me see how much the range depends on a setting I haven't validated. A correct probability calculation doesn't make the score conversion accurate.

In [ ]:
sensitivity = []
for cap in (550, 600, 650):
    r = predict_test(**modules, settings=replace(settings, easy_cap=cap),
                     include_illustrative_scores=True)
    sensitivity.append({"assumed_easy_cap": cap, **r["total_score_summary"]})
display(pd.DataFrame(sensitivity)[["assumed_easy_cap", "low", "median", "high"]])


## What I still need to test

The next notebook handles data storage and analysis. To evaluate the score model, I still need real practice-attempt inputs paired with actual scores. I would fit the conversion on one set of data and test it on separate students and test forms.

The main questions are how often the intervals contain the actual score, how wide they are, and how far the median prediction is from the actual result. I haven't measured those yet.